# SONARA Dataset V2: feature extraction with metadata and an external PVSD test set

**Run this before `SONARA_TRAIN_V6.ipynb`.** It replaces `SONARA_REVAMP_DATASET.ipynb`.

What changed from the previous notebook, and why:

| Change | Why |
| --- | --- |
| SmartEars and PVSD are saved **separately**. PVSD is never used for training. | PVSD labels come from a treated vs untreated cohort, not per-clip vet review, so it is used only as an external test. |
| File path, source, original class, duration, sample rate and an MD5 hash are saved for every sample. | Needed for per-source results, error analysis by subclass (healthy vs noise), and traceability back to the audio. |
| Audio is read with `soundfile`, not `torchaudio.load`. | Recent torchaudio versions need the separate `torchcodec` package for `load`, so the old notebook can fail on a fresh Colab runtime. |
| Short clips are padded with the clip's **minimum dB value** (deltas with 0), not with 0 dB. | 0 dB is a loud value on this scale; zero padding could create a shortcut if file lengths differ by class. |
| SmartEars clips are stored as the full 5 s (500 frames). The training notebook takes a random 4-s crop for training and the centre 4 s for validation and test. | Removes the unseeded random crop and makes evaluation deterministic. |
| Each PVSD file is cut into non-overlapping 4-s windows (up to 50 per file). | Lets the training notebook score whole files, not one random crop. |
| Inventory checks: duration by class, sample rates, exact duplicates. | Detects confounds and leakage before any model is trained. |
| `SEED` is defined. | The old notebook used `SEED` without defining it. |

**Inputs:** the two dataset folders in Google Drive, unchanged.
**Outputs** (in `OUT_DIR`): `smartears_feats.npz`, `smartears_meta.csv`, `pvsd_feats.npz`, `pvsd_windows.csv`, `pvsd_meta.csv`, `inventory.csv`, `dataset_config.json`, and two EDA figures.

Runtime: roughly 10 to 25 minutes on Colab, mostly reading audio from Drive. No GPU needed.

In [ ]:
import os, sys, json, hashlib, random, re, time
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# ---------------- CONFIG ----------------
ROOT    = os.environ.get("SONARA_ROOT", "/content/drive/MyDrive/SONARA/poultry_dataset")
OUT_DIR = os.environ.get("SONARA_OUT",  "/content/drive/MyDrive/SONARA/v2")
PVSD_DIR = os.path.join(ROOT, "Poultry Vocalization Signal Dataset for Early Disease Detection")
SMARTEARS_DIR = os.path.join(ROOT, "SmartEars A Practical Framework for Poultry Respiratory Monitoring via Spectrogram-Based Audio Classification and AI-Assisted Labeling")

SEED = 42
TARGET_SR = 16000
N_MELS = 64
N_FFT = int(0.025 * TARGET_SR)       # 25 ms window = 400 samples
HOP = int(0.010 * TARGET_SR)         # 10 ms hop = 160 samples
STORE_FRAMES = 500                   # SmartEars: keep the full 5 s
WINDOW_FRAMES = 400                  # 4-s model input
MAX_PVSD_WINDOWS = 50                # cap per PVSD file (evenly spaced if more)

# Optional: regex that extracts a recording/device ID from a SmartEars filename, so that
# clips from the same recording stay in the same split. Leave as None until you have
# inspected the filenames printed below. Example: r"^(dev\d+)_"  ->  group by device.
GROUP_PATTERN = None

# Original folder name -> (original class label, binary label: 1 = Distress)
SMARTEARS_CLASSES = {"Healthy": ("healthy", 0), "Sick": ("sick", 1), "None": ("none", 0)}
PVSD_CLASSES      = {"Healthy": ("healthy", 0), "Unhealthy": ("unhealthy", 1), "Noise": ("noise", 0)}
AUDIO_EXT = (".wav", ".flac")
# ----------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
for d in (PVSD_DIR, SMARTEARS_DIR):
    print(os.path.isdir(d), d)

## 1. Inventory every audio file

Reads only the file headers plus an MD5 hash, so it is fast. Nothing is decoded yet.

In [ ]:
def md5_of(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def inventory(base_dir, source, class_map):
    rows = []
    for folder, (orig_class, y_bin) in class_map.items():
        d = os.path.join(base_dir, folder)
        if not os.path.isdir(d):
            print("  NOT FOUND:", d)
            continue
        for fname in sorted(os.listdir(d)):
            if not fname.lower().endswith(AUDIO_EXT):
                continue
            path = os.path.join(d, fname)
            try:
                info = sf.info(path)
                rows.append(dict(filepath=path, filename=fname, source=source, orig_class=orig_class,
                                 y_bin=y_bin, sr=info.samplerate, channels=info.channels,
                                 subtype=info.subtype, duration_s=info.frames / info.samplerate,
                                 md5=md5_of(path)))
            except Exception as e:
                print("  could not read", path, ":", e)
    return pd.DataFrame(rows)

inv = pd.concat([inventory(SMARTEARS_DIR, "SmartEars", SMARTEARS_CLASSES),
                 inventory(PVSD_DIR, "PVSD", PVSD_CLASSES)], ignore_index=True)
inv.to_csv(os.path.join(OUT_DIR, "inventory.csv"), index=False)
print(inv.groupby(["source", "orig_class"]).size().rename("files"))

## 2. Checks to run before training

Read these outputs and record anything unusual in the methodology. In particular:

* **Duration by class (PVSD).** If healthy and unhealthy files differ a lot in length, a model could learn length (or the padding) instead of sound. Report this as a confound check.
* **Sample rate by class.** Different recording setups per class are another possible shortcut.
* **Exact duplicates.** Identical files in two classes are label errors; identical files within a class would leak across splits.
* **SmartEars filenames.** If names contain a device, farm or timestamp ID, set `GROUP_PATTERN` above and rerun, so clips from the same recording never land in both train and test.

In [ ]:
pd.set_option("display.width", 160)
print("Duration (s) by source and class:")
print(inv.groupby(["source", "orig_class"])["duration_s"].describe()[["count", "mean", "min", "50%", "max"]].round(2))

print("\nSample rates by source and class:")
print(inv.groupby(["source", "orig_class"])["sr"].value_counts().rename("files"))

print("\nShare of files shorter than the 4-s model input:")
print(inv.assign(short=inv.duration_s < WINDOW_FRAMES * HOP / TARGET_SR)
         .groupby(["source", "orig_class"])["short"].mean().round(3))

dup = inv[inv.duplicated("md5", keep=False)].sort_values("md5")
print(f"\nExact duplicate files: {len(dup)} files in {dup.md5.nunique()} groups")
if len(dup):
    cross = dup.groupby("md5")["orig_class"].nunique()
    print("  groups spanning more than one class (label conflicts):", int((cross > 1).sum()))
    print(dup[["source", "orig_class", "filename"]].head(20).to_string(index=False))

print("\nExample SmartEars filenames per class (look for device / time IDs):")
for c, g in inv[inv.source == "SmartEars"].groupby("orig_class"):
    print(" ", c, "->", list(g.filename.head(6)))

In [ ]:
# Grouping key for the train/val/test split (used by the training notebook).
se = inv[inv.source == "SmartEars"].copy()
if GROUP_PATTERN:
    m = se.filename.str.extract(GROUP_PATTERN, expand=False)
    print("filenames not matching GROUP_PATTERN:", int(m.isna().sum()))
    se["group"] = m.fillna(se.filename)
else:
    se["group"] = se.filename
# Exact duplicates must share a group, otherwise copies could sit in train and test.
se["group"] = se.groupby("md5")["group"].transform("first")
print("SmartEars files:", len(se), "| split groups:", se.group.nunique())
if se.groupby("md5").orig_class.nunique().max() > 1:
    print("WARNING: some identical files carry different labels; inspect them before training.")

## 3. Feature extraction

Same features as before: 16 kHz mono, 25 ms window, 10 ms hop, 64 mel bands, power in dB, plus delta and delta-delta. Output per sample: `(3, 64, frames)`.

In [ ]:
mel_spec = torchaudio.transforms.MelSpectrogram(sample_rate=TARGET_SR, n_fft=N_FFT, hop_length=HOP,
                                                n_mels=N_MELS, center=True, power=2.0)
to_db = torchaudio.transforms.AmplitudeToDB(stype="power")
deltas = torchaudio.transforms.ComputeDeltas()

def load_audio(path):
    wav, sr = sf.read(path, dtype="float32", always_2d=True)   # (samples, channels)
    wav = torch.from_numpy(wav.mean(axis=1)).unsqueeze(0)       # mono, (1, samples)
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    return wav

@torch.no_grad()
def extract_feats(path):
    db = to_db(mel_spec(load_audio(path)))          # (1, 64, T)
    d1 = deltas(db)
    d2 = deltas(d1)
    return torch.cat([db, d1, d2], dim=0)           # (3, 64, T)

def pad_to(feat, frames):
    # Pad on the right: dB channel with this clip's minimum, delta channels with 0.
    c, f, t = feat.shape
    if t >= frames:
        return feat
    out = torch.zeros(c, f, frames, dtype=feat.dtype)
    out[0] = feat[0].min()
    out[:, :, :t] = feat
    return out

def center_crop(feat, frames):
    t = feat.shape[-1]
    if t <= frames:
        return pad_to(feat, frames)
    s = (t - frames) // 2
    return feat[:, :, s:s + frames]

def pvsd_windows(feat):
    # Non-overlapping 4-s windows; a file shorter than 4 s gives one padded window.
    t = feat.shape[-1]
    if t <= WINDOW_FRAMES:
        return [(0, pad_to(feat, WINDOW_FRAMES))]
    starts = list(range(0, t - WINDOW_FRAMES + 1, WINDOW_FRAMES))
    if len(starts) > MAX_PVSD_WINDOWS:
        idx = np.linspace(0, len(starts) - 1, MAX_PVSD_WINDOWS).round().astype(int)
        starts = [starts[i] for i in idx]
    return [(s, feat[:, :, s:s + WINDOW_FRAMES]) for s in starts]

In [ ]:
t0 = time.time()
X_se = np.zeros((len(se), 3, N_MELS, STORE_FRAMES), dtype=np.float16)
se_frames = []
for i, path in enumerate(se.filepath):
    f = extract_feats(path)
    se_frames.append(f.shape[-1])
    X_se[i] = center_crop(f, STORE_FRAMES).numpy().astype(np.float16)
    if (i + 1) % 1000 == 0:
        print(f"  SmartEars {i + 1}/{len(se)}  ({time.time() - t0:.0f}s)")
se["n_frames"] = se_frames
se = se.reset_index(drop=True)
print("SmartEars features:", X_se.shape, f"{time.time() - t0:.0f}s")
print("value range check (dB channel):", float(X_se[:, 0].min()), float(X_se[:, 0].max()))

In [ ]:
pv = inv[inv.source == "PVSD"].reset_index(drop=True).copy()
win_rows, win_feats = [], []
t0 = time.time()
for i, path in enumerate(pv.filepath):
    f = extract_feats(path)
    for k, (start, w) in enumerate(pvsd_windows(f)):
        win_rows.append(dict(file_idx=i, window_idx=k, start_s=start * HOP / TARGET_SR))
        win_feats.append(w.numpy().astype(np.float16))
X_pv = np.stack(win_feats)
pv_win = pd.DataFrame(win_rows)
pv_win = pv_win.join(pv[["filepath", "filename", "orig_class", "y_bin"]], on="file_idx")
print("PVSD windows:", X_pv.shape, "from", len(pv), "files", f"({time.time() - t0:.0f}s)")
print(pv_win.groupby("orig_class").agg(files=("file_idx", "nunique"), windows=("file_idx", "size")))

In [ ]:
np.savez(os.path.join(OUT_DIR, "smartears_feats.npz"), X=X_se, y_bin=se.y_bin.values.astype(np.int64))
se.to_csv(os.path.join(OUT_DIR, "smartears_meta.csv"), index=False)
np.savez(os.path.join(OUT_DIR, "pvsd_feats.npz"), X=X_pv, y_bin=pv_win.y_bin.values.astype(np.int64))
pv_win.to_csv(os.path.join(OUT_DIR, "pvsd_windows.csv"), index=False)
pv.to_csv(os.path.join(OUT_DIR, "pvsd_meta.csv"), index=False)
cfg = dict(seed=SEED, target_sr=TARGET_SR, n_mels=N_MELS, n_fft=N_FFT, hop=HOP, store_frames=STORE_FRAMES,
           window_frames=WINDOW_FRAMES, max_pvsd_windows=MAX_PVSD_WINDOWS, group_pattern=GROUP_PATTERN,
           padding="dB channel padded with clip minimum, deltas with 0", torch=torch.__version__,
           torchaudio=torchaudio.__version__)
json.dump(cfg, open(os.path.join(OUT_DIR, "dataset_config.json"), "w"), indent=2)
print("Saved to", OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    print(f"  {f:28s} {os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6:8.1f} MB")

## 4. EDA figures for the binary task (matrix row 4)

Figure A shows one example per original subclass, titled by the **binary** label, so Non-distress is visibly varied. Figure B compares the average log-mel spectrum of Distress against each Non-distress subclass. Use these to replace Figure 3.2.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

rng = np.random.default_rng(SEED)
panels = [("sick", "Distress (SmartEars 'Sick')"),
          ("healthy", "Non-distress: healthy call"),
          ("none", "Non-distress: no chicken sound")]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
for ax, (cls, title) in zip(axes, panels):
    i = int(rng.choice(np.where(se.orig_class.values == cls)[0]))
    im = ax.imshow(X_se[i, 0].astype(np.float32), origin="lower", aspect="auto",
                   extent=[0, STORE_FRAMES * HOP / TARGET_SR, 0, N_MELS], cmap="magma")
    ax.set_title(title); ax.set_xlabel("Time (s)")
axes[0].set_ylabel("Mel band")
fig.colorbar(im, ax=axes, label="dB", shrink=0.85)
fig.savefig(os.path.join(OUT_DIR, "eda_binary_examples.png"), dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for cls, label in [("sick", "Distress"), ("healthy", "Non-distress: healthy"), ("none", "Non-distress: none")]:
    m = X_se[se.orig_class.values == cls, 0].astype(np.float32).mean(axis=(0, 2))
    ax.plot(m, label=label)
ax.set_xlabel("Mel band (low to high frequency)"); ax.set_ylabel("Mean power (dB)")
ax.set_title("Average log-mel spectrum, SmartEars"); ax.legend()
fig.savefig(os.path.join(OUT_DIR, "eda_mean_spectrum.png"), dpi=200, bbox_inches="tight")
plt.show()